<a href="https://colab.research.google.com/github/sara-sgit/Plant-Disease-Classification/blob/main/Feature__Selection_4_Genetic_ALg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install -y git


Reading package lists... Done
Building dependency tree       
Reading state information... Done
git is already the newest version (1:2.25.1-1ubuntu3.11).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.


In [ ]:
!git clone https://github.com/BiDAlab/GeneticAlgorithm.git


Cloning into 'GeneticAlgorithm'...
remote: Enumerating objects: 245, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 245 (delta 53), reused 13 (delta 13), pack-reused 190
Receiving objects: 100% (245/245), 645.17 KiB | 16.54 MiB/s, done.
Resolving deltas: 100% (126/126), done.


In [ ]:
%cd GeneticAlgorithm


/content/GeneticAlgorithm


In [ ]:
!pip install .

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Processing /content/GeneticAlgorithm
  Preparing metadata (setup.py) ... done
  Created wheel for genetic-selector: filename=genetic_selector-1.0-py3-none-any.whl size=16929 sha256=a5c173875d28b63a7f32da20dabcba53ea25f683bf3ec1f7acb97059be84d6a9
  Stored in directory: /tmp/pip-ephem-wheel-cache-0aez218l/wheels/90/4b/84/4b199d91aa88df705dd50c5d35e302532439eb7bbd45dff7f7
Successfully built genetic-selector


In [ ]:
from genetic_selector import GeneticSelector

In [ ]:
import pandas as pd
import numpy as np


In [ ]:

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from keras.models import Model
from sklearn.metrics import classification_report
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pickle

with open('/content/drive/MyDrive/BENNANI_AKRIB/Features/Densenet/Sigmoid/F_TEST_DeneseNet_sigmo.pickle', 'rb') as f:
    x_ts_densenet = pickle.load(f)
with open('MyDrive/BENNANI_AKRIB/Features/Densenet/Sigmoid/F_TRAIN_DeneseNet_sigmo.pickle', 'rb') as f:
    x_tr_densenet = pickle.load(f)

with open('MyDrive/BENNANI_AKRIB/Features/VGG/Sigmoid/F_test_VGG_SGD.pickle', 'rb') as f:
    x_ts_vgg = pickle.load(f)
with open('MyDrive/Features/VGG/Sigmoid/F_train_VGG_SGD.pickle', 'rb') as f:
    x_tr_vgg = pickle.load(f)

with open('MyDrive/BENNANI_AKRIB/Features/resnet/Sigmoid/_F_TEST_RESNET_sigmoid.pickle', 'rb') as f:
    x_ts_resnet = pickle.load(f)
with open('MyDrive/BENNANI_AKRIB/Features/resnet/Sigmoid/F_TRAIN_RESNET_sigmoid.pickle', 'rb') as f:
    x_tr_resnet = pickle.load(f)

In [ ]:
with open('MyDrive/Features/labels_test.pkl', 'rb') as f:
    labelsts = pickle.load(f)

with open('MyDrive/Features/labels_training.pkl', 'rb') as f:
    labelstr = pickle.load(f)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_normalized = scaler.fit_transform(x_tr_vgg)
X_test_normalized = scaler.transform(x_ts_vgg)

In [ ]:
from sklearn.svm import SVC
estimator = SVC(kernel="rbf", C=1)

In [ ]:
genetic_selector = GeneticSelector(
        estimator, n_gen=25, population_size=100,scoring="accuracy",
        mutation_rate=0.09,
        )

In [ ]:
genetic_selector = GeneticSelector(
    estimator=estimator,
    cv=1,  # Nombre de folds pour la validation croisée
    n_gen=10,  # Nombre de générations
    population_size=10,  # Taille de la population
    crossover_rate=0.6,  # Taux de crossover
    mutation_rate=0.09,  # Taux de mutation
    tournament_k=2,  # Taille du tournoi pour la sélection
    scoring="accuracy",  # Métrique de performance à optimiser
    calc_train_score=True,  # Calculer le score sur les données d'entraînement
    #initial_best_chromosome=best_chromosome,  # Meilleur chromosome initial (facultatif)
    n_jobs=-1,  # Utiliser tous les cœurs disponibles
    #random_state=random_state,  # Graine aléatoire pour la reproductibilité
    verbose=0  # Niveau de verbosité
)

In [ ]:
type(X_train_normalized)

numpy.ndarray

In [ ]:
X_train_normalized_1=pd.DataFrame(X_train_normalized)

In [ ]:
model = genetic_selector.fit(X_train_normalized_1, labelstr)

# Creating initial population with 10 chromosomes...
# Evaluating initial population...


# sklearn-genetic package

[Lien](https://pythonguides.com/scikit-learn-genetic-algorithm/)

[Lien vers l'explication des attributs](https://sklearn-genetic-opt.readthedocs.io/en/stable/api/gafeatureselectioncv.html#sklearn_genetic.GAFeatureSelectionCV)

In [ ]:
!pip install sklearn-genetic

In [ ]:
from sklearn.svm import SVC
estimator = SVC(kernel="rbf", C=1)

In [ ]:
model = GeneticSelectionCV(
    estimator, cv=5, verbose=0,
    scoring="accuracy", max_features=100,
    n_population=100, crossover_proba=0.6,n_jobs=-1,
    mutation_proba=0.09, n_generations=2,)

In [ ]:
model = model.fit(X_train_normalized, labelstr)

In [ ]:
X1_vgg =model.transform(X_train_normalized)
X2_vgg =model.transform(X_test_normalized)

In [ ]:
from sklearn.pipeline import make_pipeline

svm = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1, gamma='scale'))
# Fit the model to the training data on the GPU
svm.fit(X1_vgg.astype(np.float32), labelstr.astype(np.float32))
# Predict the labels of the test data
y_pred = svm.predict(X2_vgg .astype(np.float32))

#________________________________   #########   #matrice de confusion    #############_____________________________________________________________

from sklearn.metrics import classification_report, confusion_matrix

print('confusion matrix')
print(confusion_matrix(labelsts,y_pred))
labels=labelsts
sum((labels==y_pred)*1)/len(y_pred)